In [17]:
def correct_date(row):

    if row["FECHA"][0] == "'":
        row["FECHA"] = row["FECHA"][1:]
    if "/" in row["FECHA"]:
        date_split = row["FECHA"].split("/")
    if "-" in row["FECHA"]:
        date_split = row["FECHA"].split("-")
    try:
        day = date_split[0].zfill(2)
        month = date_split[1].zfill(2)
        all_date = day + "/" + month + "/" + date_split[2]
        return all_date
    except Exception as e:
        raise ValueError(f"Error processing date: {row['FECHA']}. Error: {e}")


In [26]:
import pandas as pd
import os

main = os.getcwd().replace("\\", "/").split("notebooks")[0]

all_df = pd.read_excel(main + "data/facturas_ia+parser.xlsx")

duplicados = all_df[all_df.duplicated("FACTURA", keep=False)]
no_duplicados = all_df[~all_df.duplicated("FACTURA", keep=False)]

dup_ord = duplicados.sort_values(by=["EXTRACTION"], ascending=False)

dup_ord = dup_ord.dropna(subset=["FACTURA"])

res = dup_ord.drop_duplicates(subset=["FACTURA"], keep="first")

final = pd.concat([res, no_duplicados], ignore_index=True)

final = final[final["CHECK"] != "Sin nombre"]

final = final[["FECHA", "PROVEEDOR", "NIT", "FACTURA","VALOR ANTES DE IVA", "IVA", "TOTAL", "CHECK"]]
final = final.dropna()
final["FECHA"] = final.apply(lambda row: correct_date(row)  , axis=1)
final["FECHA"] = pd.to_datetime(final["FECHA"], format="mixed", dayfirst=True).dt.strftime("%d/%m/%Y")


In [27]:
final.to_excel(main + "data/iva_armando.xlsx", index=False)